# Mask-Aware and Safety-Aware Adaptive Weighting Extension
## LSTM Graph Laplacian Regularized Matrix Factorization for Seattle Speed Data

> **Extended from:** LSTM-GL-ReMF (Jinming Yang, SJTU)  
> **Novel contribution:** Mask-aware + Safety-aware adaptive weighting for real-time CAV trajectory imputation at edge servers.

### What changed from the original?
The original model used **fixed global weights** `lambda_x` and `lambda_g` throughout training. These scalars treat every sensor, every time step, and every missing pattern identically.

This extension replaces those fixed weights with a **local adaptive gating mechanism** that:
1. **Mask-aware gate**: Reads the local missingness context (gap duration, missing ratio, neighbor availability) and dynamically shifts reliance between the temporal LSTM branch and spatial graph branch.
2. **Safety-aware adjustment**: Penalizes branches whose candidate reconstructions violate kinematic plausibility (acceleration/jerk bounds, speed bounds), reducing their influence before final fusion.

All other model structure (ALS solver, LSTM backbone, graph Laplacian) is **retained unchanged**.

## Data Organization: Matrix Structure

We consider a dataset of $M$ discrete time series $\boldsymbol{y}_{i}\in\mathbb{R}^{T},i\in\left\{1,2,...,M\right\}$. The time series may have missing elements. We express spatio-temporal dataset as a matrix $Y\in\mathbb{R}^{M\times T}$.

## Adaptive Weighting Formulation

At each vehicle $i$ and time step $t$, the imputed value is:
$$\hat{y}_{it} = \alpha_{it} \cdot \hat{y}^{\text{temp}}_{it} + (1 - \alpha_{it}) \cdot \hat{y}^{\text{spatial}}_{it}$$

where $\alpha_{it} \in [0,1]$ is produced by a lightweight MLP gate taking mask-aware features as input, further adjusted by a safety confidence score:
$$\alpha_{it}^{\text{safe}} = \alpha_{it} \cdot c^{\text{temp}}_{it} / (\alpha_{it} \cdot c^{\text{temp}}_{it} + (1-\alpha_{it}) \cdot c^{\text{spatial}}_{it})$$

where $c^{\text{temp}}_{it}$ and $c^{\text{spatial}}_{it}$ are kinematic plausibility confidence scores in $[0,1]$.

In [ ]:
import os
import numpy as np
import time

CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(W, X, model, gate_model, iters, rmse, mape):
    """
    MODIFIED: Now also saves the adaptive gate model weights.
    The gate model is a small MLP trained alongside the main model.
    """
    np.savez(
        os.path.join(CHECKPOINT_DIR, "mf_checkpoint.npz"),
        W=W,
        X=X,
        iters=iters,
        rmse=rmse,
        mape=mape
    )
    model.save_weights(os.path.join(CHECKPOINT_DIR, "lstm_weights.h5"))
    # NEW: Save gate model weights
    if gate_model is not None:
        gate_model.save_weights(os.path.join(CHECKPOINT_DIR, "gate_weights.h5"))
    print(f"✅ Checkpoint saved at iteration {iters}")


def load_checkpoint(model, gate_model=None):
    """
    MODIFIED: Also restores gate model weights if available.
    """
    ckpt_path = os.path.join(CHECKPOINT_DIR, "mf_checkpoint.npz")
    weights_path = os.path.join(CHECKPOINT_DIR, "lstm_weights.h5")

    if os.path.exists(ckpt_path) and os.path.exists(weights_path):
        data = np.load(ckpt_path)
        model.load_weights(weights_path)
        # NEW: Try loading gate weights
        gate_path = os.path.join(CHECKPOINT_DIR, "gate_weights.h5")
        if gate_model is not None and os.path.exists(gate_path):
            gate_model.load_weights(gate_path)
            print("✅ Gate model weights restored.")
        print(f"✅ Checkpoint loaded (resuming from iteration {int(data['iters'])})")
        return (
            data["W"],
            data["X"],
            int(data["iters"]),
            float(data["rmse"]),
            float(data["mape"]),
        )

    print("⚠️ No checkpoint found. Starting fresh.")
    return None

In [ ]:
from numpy.linalg import inv as inv
from keras.models import Sequential, Model
from keras.layers import Dense, LSTM, Dropout, Input
from keras.optimizers import Adam
import tensorflow as tf

tf.config.list_physical_devices('GPU')

In [ ]:
!mkdir -p datasets/Seattle_loop-data-set
!wget https://raw.githubusercontent.com/yangjm67/TransPAI/master/datasets/Seattle_loop-data-set/dense_mat.npy -P datasets/Seattle_loop-data-set
!wget https://raw.githubusercontent.com/yangjm67/TransPAI/master/datasets/Seattle_loop-data-set/Loop_Seattle_2015_A.npy -P datasets/Seattle_loop-data-set
!wget https://raw.githubusercontent.com/yangjm67/TransPAI/master/datasets/Seattle_loop-data-set/nm_random_mat.npy -P datasets/Seattle_loop-data-set

In [ ]:
directory = 'datasets/Seattle_loop-data-set/'
ADJ = np.load(directory + 'Loop_Seattle_2015_A.npy')
dense_mat = np.load(directory + 'dense_mat.npy')
print('Adjacency matrix shape:', ADJ.shape)
print('Dataset shape:', dense_mat.shape)

missing_rate = 0.4
# Non-random missing (CM) scenario
nm_random_mat = np.load(directory + 'nm_random_mat.npy')
binary_tensor = np.zeros((dense_mat.shape[0], 61, 288))
for i1 in range(binary_tensor.shape[0]):
    for i2 in range(binary_tensor.shape[1]):
        binary_tensor[i1, i2, :] = np.round(nm_random_mat[i1, i2] + 0.5 - missing_rate)
binary_mat = binary_tensor.reshape([binary_tensor.shape[0], binary_tensor.shape[1] * binary_tensor.shape[2]])

sparse_mat = np.multiply(dense_mat, binary_mat)

In [ ]:
test_rate = 0.082
train_len = int((1 - test_rate) * sparse_mat.shape[1])
test_len = sparse_mat.shape[1] - train_len
training_set = sparse_mat[:, :train_len]
test_set = sparse_mat[:, train_len:]
training_ground_truth = dense_mat[:, :train_len]
test_ground_truth = dense_mat[:, train_len:]
print('Training set shape:', training_set.shape)
print('Test set shape:', test_set.shape)

## Utility Functions (unchanged from original)

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred, pos): 
    return np.mean(np.abs((y_true[pos] - y_pred[pos]) / y_true[pos])) * 100

def root_mean_squared_error(y_true, y_pred, pos): 
    return np.sqrt(np.mean(np.square(y_true[pos] - y_pred[pos])))

def create_lstm_samples(dataset, time_lags, rate):
    dataX, dataY = [], []
    data_len = dataset.shape[0] - np.max(time_lags)
    t_sample = np.random.choice(data_len, int(rate * data_len), replace=False)
    for t in t_sample:
        a = dataset[t + np.max(time_lags) - time_lags, :][::-1]
        dataX.append(a)
        dataY.append(dataset[t + np.max(time_lags), :])
    return np.array(dataX), np.array(dataY)

def lstmmodel(rank, lag_len):
    """UNCHANGED from original: temporal LSTM backbone."""
    model = Sequential()
    model.add(LSTM(rank, input_shape=(lag_len, rank)))
    model.add(Dense(rank))
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model

## NEW: Mask-Aware Feature Extractor

### Why this is needed
The original model applied the same `lambda_x` (temporal weight) and `lambda_g` (spatial weight) everywhere. 
But in real CAV settings:
- A **short gap** (1-2 time steps) → temporal continuity is very reliable → prefer temporal branch
- A **long burst** of missing data → temporal estimate degrades → prefer spatial neighbors
- **No neighbors available** → can only trust temporal branch

This function extracts cheap, interpretable features from the local mask context that the gate MLP uses to compute $\alpha_{it}$.

In [ ]:
def compute_mask_features(binary_mat, ADJ, t, window=30):
    """
    NEW FUNCTION: Compute mask-aware features for the adaptive gate.
    
    These features are computed per-timestep (not per sensor) for efficiency,
    which is appropriate for the temporal embedding update in ALS.
    
    Args:
        binary_mat : (M, T) binary mask (1=observed, 0=missing)
        ADJ        : (M, M) adjacency matrix
        t          : current time step index
        window     : look-back window in time steps (default=30, ~3s at 10Hz)
    
    Returns:
        features : np.array of shape (5,) with values in [0, 1]
    
    Feature breakdown:
        [0] missing_ratio      : fraction of sensors missing at time t
        [1] gap_since_last     : normalised time steps since last fully observed step
        [2] burst_length       : normalised length of current consecutive missing run
        [3] neighbor_avail     : fraction of graph edges with both endpoints observed
        [4] recent_volatility  : std of missing ratio over past `window` steps (normalised)
    """
    dim1, dim2 = binary_mat.shape
    col = binary_mat[:, t]  # (M,) observation mask at time t
    
    # Feature 0: fraction of sensors missing at t
    missing_ratio = 1.0 - col.mean()
    
    # Feature 1: steps since the last time step where >50% sensors were observed
    gap = 0
    for tau in range(t - 1, max(t - window, -1), -1):
        if binary_mat[:, tau].mean() >= 0.5:
            break
        gap += 1
    gap_since_last = min(gap / window, 1.0)
    
    # Feature 2: length of current consecutive missing run at time t
    burst = 0
    for tau in range(t, max(t - window, -1), -1):
        if binary_mat[:, tau].mean() >= 0.5:  # majority observed → burst ended
            break
        burst += 1
    burst_length = min(burst / window, 1.0)
    
    # Feature 3: fraction of graph edges where both endpoints are observed
    src, dst = np.where(ADJ == 1)
    if len(src) > 0:
        both_observed = (col[src] * col[dst]).mean()
    else:
        both_observed = 0.0
    neighbor_avail = float(both_observed)
    
    # Feature 4: volatility (std) of missing ratio in the recent window
    t_start = max(0, t - window)
    recent_missing = 1.0 - binary_mat[:, t_start:t+1].mean(axis=0)  # shape (window,)
    volatility = float(np.std(recent_missing)) * 2.0  # scale ~[0,1]
    volatility = min(volatility, 1.0)
    
    return np.array([missing_ratio, gap_since_last, burst_length, neighbor_avail, volatility],
                    dtype=np.float32)

## NEW: Lightweight Adaptive Gate MLP

### Why this architecture?
The paper specifies a **lightweight** gating network for edge deployment. A 2-layer MLP with 16 hidden units and sigmoid output:
- Takes 5 mask-aware features as input
- Outputs $\alpha \in [0,1]$ — weight for temporal branch  
- $(1 - \alpha)$ automatically becomes the spatial branch weight
- ~500 parameters total → fits within edge timing budget

In [ ]:
def build_gate_model(n_features=5, hidden=16):
    """
    NEW FUNCTION: Lightweight MLP gate that outputs alpha in [0, 1].
    
    alpha → weight given to temporal branch.
    (1 - alpha) → weight given to spatial (graph) branch.
    
    Architecture: Linear(5→16) → ReLU → Linear(16→1) → Sigmoid
    ~500 parameters — negligible edge overhead.
    """
    inp = Input(shape=(n_features,))
    h = Dense(hidden, activation='relu')(inp)
    out = Dense(1, activation='sigmoid')(h)
    model = Model(inputs=inp, outputs=out)
    model.compile(loss='mse', optimizer=Adam(learning_rate=1e-3))
    return model


def predict_alpha(gate_model, mask_features):
    """
    NEW FUNCTION: Run gate forward pass. Returns scalar alpha in [0,1].
    """
    feat = mask_features[np.newaxis, :]  # (1, 5)
    return float(gate_model.predict(feat, verbose=0)[0, 0])

## NEW: Safety-Aware Kinematic Confidence Scores

### Why safety checks?
In a CAV edge system, imputed trajectory histories feed directly into **downstream prediction and control**. A branch that produces a physically implausible reconstruction (e.g., acceleration of 20 m/s²) must be down-weighted before fusion — regardless of what the data-driven model says.

The safety check computes a confidence score $c \in [0,1]$ for each candidate reconstruction:
- $c = 1.0$ → fully plausible (no constraint violations)
- $c \to 0$ → severe kinematic violations

These scores multiplicatively adjust $\alpha$ before final fusion.

In [ ]:
# Kinematic bounds (configurable — set for road speed in km/h at 5-min intervals)
# Adjust to 10Hz if using trajectory data: dt=0.1s, max_acc ~3 m/s² ≈ 10.8 km/h per s
SPEED_MIN   = 0.0    # km/h (cannot be negative)
SPEED_MAX   = 130.0  # km/h (highway upper bound)
ACC_MAX     = 40.0   # km/h per time-step — generous for 5-min loop-detector data
JERK_MAX    = 20.0   # km/h per time-step change in acceleration


def kinematic_confidence(y_candidate, y_prev=None, y_pprev=None):
    """
    NEW FUNCTION: Compute a kinematic plausibility confidence score in [0, 1].
    
    A score of 1.0 means no violations. Each violation reduces the score
    by a multiplicative penalty so scores compose gracefully.
    
    Args:
        y_candidate : (M,) imputed speed vector for sensors at time t
        y_prev      : (M,) imputed speed at t-1 (or None)
        y_pprev     : (M,) imputed speed at t-2 (for jerk, or None)
    
    Returns:
        confidence  : float in [0, 1]
    
    Why this design?
        - Multiplicative penalties mean a single minor violation gives ~0.9,
          a severe violation gives ~0.1-0.3. This is interpretable and tunable.
        - Fraction-of-sensors approach: penalty scales with how many sensors
          violate, not binary pass/fail.
    """
    confidence = 1.0
    M = len(y_candidate)
    
    # Check 1: Speed bounds
    frac_speed_viol = np.mean((y_candidate < SPEED_MIN) | (y_candidate > SPEED_MAX))
    confidence *= max(0.1, 1.0 - frac_speed_viol)
    
    if y_prev is not None:
        # Check 2: Acceleration bound |v_t - v_{t-1}| < ACC_MAX
        acc = np.abs(y_candidate - y_prev)
        frac_acc_viol = np.mean(acc > ACC_MAX)
        confidence *= max(0.1, 1.0 - frac_acc_viol)
    
    if y_prev is not None and y_pprev is not None:
        # Check 3: Jerk bound |a_t - a_{t-1}| < JERK_MAX
        acc_t   = y_candidate - y_prev
        acc_tm1 = y_prev - y_pprev
        jerk = np.abs(acc_t - acc_tm1)
        frac_jerk_viol = np.mean(jerk > JERK_MAX)
        confidence *= max(0.1, 1.0 - frac_jerk_viol)
    
    return float(confidence)


def safety_adjusted_alpha(alpha, c_temp, c_spatial):
    """
    NEW FUNCTION: Adjust gate output alpha using safety confidence scores.
    
    Implements the formula from the paper:
        alpha_safe = (alpha * c_temp) / (alpha * c_temp + (1-alpha) * c_spatial)
    
    If both branches are equally unsafe, alpha passes through unchanged.
    If the temporal branch is unsafe, alpha decreases (spatial preferred).
    If the spatial branch is unsafe, alpha increases (temporal preferred).
    """
    num = alpha * c_temp
    den = num + (1.0 - alpha) * c_spatial
    if den < 1e-8:  # degenerate: both branches fully implausible
        return alpha  # fall back to mask-aware weight
    return float(num / den)

## MODIFIED: LSTM-GL-ReMF Training with Adaptive Weighting

### Summary of changes vs original `LSTM_GL_ReMF`:

| Component | Original | Modified |
|---|---|---|
| Temporal weight in X update | fixed `lambda_x` | `lambda_x * alpha_t` (adaptive) |
| Spatial weight in W update | fixed `lambda_g` | `lambda_g * (1 - alpha_t)` (adaptive) |
| Gate | none | 5-feature MLP, trained end-to-end |
| Safety check | none | kinematic confidence scores adjust alpha |
| Checkpoint | W, X, LSTM | W, X, LSTM, gate MLP |

The ALS closed-form updates are rederived to absorb $\alpha$ and $(1-\alpha)$ as scalar multipliers on the regularization terms. This keeps the update **still closed-form** — no numerical optimiser needed for the ALS step.

In [ ]:
def LSTM_GL_ReMF_Adaptive(
    sparse_mat, ADJ, init, time_lags,
    lambda_w, lambda_g, lambda_x, eta,
    sampling_rate, maxiter, track,
    patience=5, dense_mat=0, resume=True,
    # NEW hyperparameters:
    gate_train_freq=10,   # retrain gate every N iterations
    gate_epochs=5,        # epochs per gate training step
    window=30             # mask feature look-back window
):
    """
    MODIFIED VERSION of LSTM_GL_ReMF.
    
    NEW ARGS:
        gate_train_freq : how often (in ALS iterations) to retrain the gate MLP.
                          Retraining every step is too expensive; every 10 is a
                          good trade-off between adaptation speed and overhead.
        gate_epochs     : epochs to run on gate training steps.
        window          : look-back for mask feature computation (time steps).
    
    DESIGN RATIONALE for adaptive update of X:
        Original X update (t >= max_lags):
            x_t = (WtT Wt + lambda_x I + lambda_x*eta I)^{-1} (WtT y_t + lambda_x Qt)
        
        With adaptive weighting, lambda_x is scaled by alpha_t \in [0,1]:
            effective_lambda_x = lambda_x * alpha_t
        When alpha_t=1 → full reliance on LSTM temporal estimate (same as original).
        When alpha_t=0 → LSTM regularisation zeroed out, only data + L2 regulariser survive.

    DESIGN RATIONALE for adaptive update of W:
        Original W update:
            w_i = (XtT Xt + lambda_w*eta I + |N_i| * lambda_g I)^{-1} (...)
        
        With adaptive weighting, lambda_g is scaled by (1 - alpha_t):
            effective_lambda_g = lambda_g * (1 - alpha_t)
        When alpha_t=0 → spatial graph regularisation at full strength.
        When alpha_t=1 → graph term zeroed, spatial neighbors don't contribute.
        
        alpha_t here is the time-averaged alpha over the current iteration,
        since W is a sensor-level embedding (not time-specific).
    """
    dim1, dim2 = sparse_mat.shape
    binary_mat = np.zeros((dim1, dim2))
    position = np.where((sparse_mat != 0))
    binary_mat[position] = 1

    d = len(time_lags)
    max_lags = np.max(time_lags)
    r = init["X"].shape[1]

    # ---------- Build models ----------
    model = lstmmodel(r, d)          # UNCHANGED: temporal LSTM backbone
    gate_model = build_gate_model()  # NEW: lightweight adaptive gate

    # ---------- Resume logic ----------
    start_iter = 0
    rmse_pre = float(np.inf)
    mape_pre = float(np.inf)

    if resume:
        ckpt = load_checkpoint(model, gate_model)
        if ckpt is not None:
            W, X, start_iter, rmse_pre, mape_pre = ckpt
        else:
            W = init["W"]
            X = init["X"]
    else:
        W = init["W"]
        X = init["X"]

    # ---------- Tracking ----------
    if track:
        pos_err = np.where((sparse_mat == 0) & (dense_mat != 0))
        count = 0

    start_time = time.time()

    # NEW: Buffers to accumulate gate training data across the iteration
    gate_feat_buffer = []   # mask features (X)
    gate_label_buffer = []  # alpha labels — we use reconstruction error as supervision

    # NEW: Track per-iteration alpha for logging and W-update
    alpha_history = []  # one value per time step

    # ---------- Training loop ----------
    for iters in range(start_iter, maxiter):
        alpha_per_t = np.zeros(dim2)  # alpha_t for each time step this iteration

        # ============================================================
        # UPDATE X (temporal embeddings) — MODIFIED
        # ============================================================
        # We keep a rolling reconstruction for safety checks:
        Y_hat_prev  = np.matmul(W, X[max(0, 0), :])  # placeholder
        Y_hat_pprev = Y_hat_prev.copy()

        for t in range(dim2):
            pos0 = np.where(sparse_mat[:, t] != 0)
            Wt = W[pos0[0], :]

            if iters == 0 or t < max_lags:
                # No temporal LSTM estimate available yet — use vanilla ALS (UNCHANGED)
                X[t, :] = np.matmul(
                    inv(np.matmul(Wt.T, Wt) + lambda_x * eta * np.eye(r)),
                    np.matmul(Wt.T, sparse_mat[pos0[0], t])
                )
                alpha_per_t[t] = 0.5  # neutral weight before gate is trained

            else:
                # --- Temporal candidate ---
                X_hat = X[t - time_lags, :][::-1]
                X_hat_feed = X_hat[np.newaxis, :, :]
                Qt = model.predict(X_hat_feed, verbose=0)[0]  # LSTM prediction of x_t
                
                # Temporal imputation candidate: y_temp = W @ Qt
                y_temp = np.matmul(W, Qt)
                y_temp = np.clip(y_temp, SPEED_MIN, None)  # non-negative speeds

                # --- Spatial candidate ---
                # The spatial candidate is the current MF reconstruction using only data
                # and L2 regularization (no LSTM term), which represents spatial consensus
                X_spatial = np.matmul(
                    inv(np.matmul(Wt.T, Wt) + lambda_x * eta * np.eye(r)),
                    np.matmul(Wt.T, sparse_mat[pos0[0], t])
                )
                y_spatial = np.matmul(W, X_spatial)
                y_spatial = np.clip(y_spatial, SPEED_MIN, None)

                # -----------------------------------------------
                # NEW: Compute mask-aware gate alpha
                # -----------------------------------------------
                mask_feats = compute_mask_features(binary_mat, ADJ, t, window=window)
                alpha = predict_alpha(gate_model, mask_feats)  # alpha in [0,1]

                # -----------------------------------------------
                # NEW: Compute safety confidence scores
                # -----------------------------------------------
                y_prev  = Y_hat_prev  if t >= 1 else None
                y_pprev = Y_hat_pprev if t >= 2 else None

                c_temp    = kinematic_confidence(y_temp,    y_prev, y_pprev)
                c_spatial = kinematic_confidence(y_spatial, y_prev, y_pprev)

                # -----------------------------------------------
                # NEW: Safety-adjusted alpha
                # -----------------------------------------------
                alpha_safe = safety_adjusted_alpha(alpha, c_temp, c_spatial)
                alpha_per_t[t] = alpha_safe

                # -----------------------------------------------
                # MODIFIED X update:
                # effective_lambda_x = lambda_x * alpha_safe
                # This replaces the fixed lambda_x in the temporal regulariser.
                # -----------------------------------------------
                eff_lx = lambda_x * alpha_safe

                X[t, :] = np.matmul(
                    inv(
                        np.matmul(Wt.T, Wt)
                        + eff_lx * np.eye(r)
                        + lambda_x * eta * np.eye(r)
                    ),
                    np.matmul(Wt.T, sparse_mat[pos0[0], t]) + eff_lx * Qt
                )

                # Update rolling reconstruction for next step's safety check
                Y_hat_pprev = Y_hat_prev.copy()
                Y_hat_prev  = np.matmul(W, X[t, :])

                # -----------------------------------------------
                # NEW: Accumulate gate training labels
                # Label = 1.0 if temporal branch was more accurate than spatial,
                #         0.0 otherwise. Only possible when ground truth available.
                # During training with track=True we have dense_mat.
                # -----------------------------------------------
                if track and isinstance(dense_mat, np.ndarray):
                    y_true_t = dense_mat[:, t]
                    err_temp    = np.mean(np.abs(y_temp    - y_true_t))
                    err_spatial = np.mean(np.abs(y_spatial - y_true_t))
                    # Soft label: how much better is temporal?
                    label = 1.0 / (1.0 + np.exp(err_temp - err_spatial))  # sigmoid
                    gate_feat_buffer.append(mask_feats)
                    gate_label_buffer.append([label])

        # ============================================================
        # UPDATE W (spatial embeddings) — MODIFIED
        # ============================================================
        # Use the mean alpha across the iteration as the effective spatial weight
        # Why mean? W is a global sensor embedding (not time-specific), so we use
        # the average temporal preference across all time steps as the global signal.
        alpha_mean = float(np.mean(alpha_per_t))
        eff_lg = lambda_g * (1.0 - alpha_mean)  # MODIFIED: adaptive spatial weight

        for i in range(dim1):
            pos0 = np.where(sparse_mat[i, :] != 0)
            pos1 = np.where(ADJ[i, :] == 1)[0]

            vec1 = np.sum(W[pos1, :], axis=0)
            Xt = X[pos0[0], :]
            vec0 = np.matmul(Xt.T, sparse_mat[i, pos0[0]]) + eff_lg * vec1  # MODIFIED

            mat0 = inv(
                np.matmul(Xt.T, Xt)
                + lambda_w * eta * np.eye(r)
                + len(pos1) * eff_lg * np.eye(r)  # MODIFIED
            )
            W[i, :] = np.matmul(mat0, vec0)

        # ============================================================
        # TRAIN LSTM backbone — UNCHANGED
        # ============================================================
        if iters == 0:
            lstmX, lstmY = create_lstm_samples(X, time_lags, 1)
            model.fit(lstmX, lstmY, epochs=20, batch_size=50, verbose=0)
        else:
            lstmX, lstmY = create_lstm_samples(X, time_lags, sampling_rate)
            model.fit(lstmX, lstmY, epochs=2, batch_size=200, verbose=0)

        # ============================================================
        # NEW: TRAIN GATE MLP (every gate_train_freq iterations)
        # ============================================================
        if iters > 0 and (iters % gate_train_freq == 0) and len(gate_feat_buffer) > 32:
            gX = np.array(gate_feat_buffer)
            gY = np.array(gate_label_buffer)
            gate_model.fit(gX, gY, epochs=gate_epochs, batch_size=64, verbose=0)
            # Clear buffer to focus on recent data (sliding window supervision)
            gate_feat_buffer.clear()
            gate_label_buffer.clear()

        # ============================================================
        # Logging + checkpoint — MODIFIED to include gate
        # ============================================================
        if (iters + 1) % 50 == 0:
            print(f"Iterations: {iters + 1}, time cost: {int(time.time() - start_time)}s")
            print(f"  Mean alpha (temporal weight): {alpha_mean:.3f}")
            print(f"  Effective spatial lambda_g:   {eff_lg:.2f}")
            start_time = time.time()

            if track:
                mat_hat = np.matmul(W, X.T)
                mat_hat[position] = sparse_mat[position]
                mat_hat[mat_hat < 0] = 0

                rmse = root_mean_squared_error(dense_mat, mat_hat, pos_err)
                mape = mean_absolute_percentage_error(dense_mat, mat_hat, pos_err)
                print(f"  Imputation RMSE = {rmse:.2f}")
                print(f"  Imputation MAPE = {mape:.2f}")

                if (rmse_pre - rmse) < 0.001 and (mape_pre - mape) < 0.001:
                    count += 1
                    if count == patience:
                        print(f"Early stopping at iteration {iters + 1}")
                        break
                else:
                    count = 0

                rmse_pre = rmse
                mape_pre = mape

            save_checkpoint(W, X, model, gate_model, iters + 1, rmse_pre, mape_pre)
            print()

    # ---------- Final reconstruction ----------
    mat_hat = np.matmul(W, X.T)
    mat_hat[position] = sparse_mat[position]
    mat_hat[mat_hat < 0] = 0

    return mat_hat, W, X, model, gate_model

## MODIFIED: Online Temporal Embedding Calibration

### Change vs original `OnlineLSTMReMF`:
The online function now accepts an optional `gate_model` and `binary_history` (recent mask).
When present, it computes adaptive alpha and applies it to the regularization strength,
exactly mirroring the training-time logic.

In [ ]:
def OnlineLSTMReMF_Adaptive(
    sparse_vec, init, lambda_x, time_lags,
    # NEW optional arguments:
    gate_model=None, binary_history=None, ADJ=None,
    t_idx=0, y_prev=None, y_pprev=None
):
    """
    MODIFIED VERSION of OnlineLSTMReMF.
    
    NEW ARGS:
        gate_model      : trained gate MLP (None → falls back to original behaviour)
        binary_history  : (M, window) recent observation mask for feature computation
        ADJ             : adjacency matrix for neighbor availability feature
        t_idx           : current time step index (for feature computation)
        y_prev          : (M,) previous imputed speed for safety check
        y_pprev         : (M,) two steps back for jerk check
    """
    time_lags = time_lags[::-1]
    W = init["W"]
    X = init["X"]
    model = init["model"]
    t, rank = X.shape

    # Temporal LSTM estimate (UNCHANGED)
    X_hat = X[t - 1 - time_lags, :].copy()
    X_hat_feed = X_hat[np.newaxis, :, :]
    Qt = model.predict(X_hat_feed)[0]

    pos0 = np.where(sparse_vec != 0)
    Wt = W[pos0[0], :]

    # Default: fixed lambda_x (original behaviour)
    eff_lx = lambda_x
    alpha_val = 0.5

    # NEW: Apply adaptive gate if available
    if gate_model is not None and binary_history is not None and ADJ is not None:
        mask_feats = compute_mask_features(binary_history, ADJ, 
                                            min(t_idx, binary_history.shape[1] - 1),
                                            window=min(30, binary_history.shape[1]))
        alpha_val = predict_alpha(gate_model, mask_feats)

        # Safety check on temporal candidate
        y_temp = np.matmul(W, Qt)
        c_temp = kinematic_confidence(y_temp, y_prev, y_pprev)

        # Spatial candidate (data-only MF estimate)
        if Wt.shape[0] > 0:
            X_spatial = np.matmul(
                inv(np.matmul(Wt.T, Wt) + lambda_x * 0.2 * np.eye(rank)),
                np.matmul(Wt.T, sparse_vec[pos0])
            )
            y_spatial = np.matmul(W, X_spatial)
            c_spatial = kinematic_confidence(y_spatial, y_prev, y_pprev)
        else:
            c_spatial = 1.0  # no data, can't evaluate

        # Safety-adjusted alpha
        alpha_val = safety_adjusted_alpha(alpha_val, c_temp, c_spatial)
        eff_lx = lambda_x * alpha_val

    # ALS update — structure UNCHANGED, only eff_lx may differ
    var_mu = np.matmul(Wt.T, sparse_vec[pos0]) + eff_lx * Qt
    inv_var_Lambda = inv(np.matmul(Wt.T, Wt) + eff_lx * np.eye(rank))
    return np.matmul(inv_var_Lambda, var_mu), alpha_val

## MODIFIED: Online Prediction Framework

### Changes:
- Accepts `gate_model` and `ADJ` from the training phase
- Maintains a rolling `binary_history` buffer for mask feature computation
- Passes `y_prev`/`y_pprev` for safety checks at each step
- Logs mean alpha over the test set for interpretability

In [ ]:
def online_prediction_adaptive(
    sparse_mat, init, time_lags, lambda_w, lambda_x, eta, maxiter,
    # NEW:
    gate_model=None, ADJ=None, window=30
):
    """
    MODIFIED VERSION of online_prediction.
    
    Adds adaptive gate and safety-aware imputation during online rollout.
    All prediction logic unchanged; only the OnlineLSTMReMF call is updated.
    """
    W = init["W"]
    X = init["X"]
    model = init["model"]
    pre_step_num = X.shape[0]
    rank = X.shape[1]
    dim1, dim2 = sparse_mat.shape

    X_hat = np.zeros((dim2 + pre_step_num, rank))
    mat_pred = np.zeros((dim1, dim2))
    X_hat[:pre_step_num, :] = X.copy()

    # NEW: binary history buffer — (M, window) rolling window
    bin_buf_size = window + 10
    binary_history = np.zeros((dim1, bin_buf_size))

    # NEW: rolling speed for safety checks
    y_prev  = None
    y_pprev = None

    alpha_log = []  # NEW: log alpha values for analysis
    start_time = time.time()

    for t in range(dim2):
        # NEW: Update binary history buffer
        obs_col = (sparse_mat[:, t] != 0).astype(float)
        binary_history = np.roll(binary_history, -1, axis=1)
        binary_history[:, -1] = obs_col

        if t == 0:
            # First step: no previous observation to calibrate — just predict
            X_star = X_hat[pre_step_num + t - time_lags, :][::-1]
            X_star_feed = X_star[np.newaxis, :, :]
            Qt = model.predict(X_star_feed)[0]
            X_hat[pre_step_num + t, :] = Qt.copy()
            alpha_log.append(0.5)
        else:
            sparse_vec = sparse_mat[:, t - 1]
            if np.where(sparse_vec > 0)[0].shape[0] > 0:
                init_step = {
                    "W": W,
                    "X": X_hat[pre_step_num + t - np.max(time_lags) - 1: pre_step_num + t, :],
                    "model": model
                }
                # MODIFIED: pass gate and safety context
                X_c, alpha_val = OnlineLSTMReMF_Adaptive(
                    sparse_vec, init_step, lambda_x / dim2, time_lags,
                    gate_model=gate_model,
                    binary_history=binary_history,
                    ADJ=ADJ,
                    t_idx=bin_buf_size - 1,
                    y_prev=y_prev,
                    y_pprev=y_pprev
                )
                alpha_log.append(alpha_val)
                X_hat[pre_step_num + t - 1, :] = X_c.copy()

                # NEW: Update rolling speed history
                y_pprev = y_prev
                y_prev  = np.matmul(W, X_c)

                X_star = X_hat[pre_step_num + t - time_lags, :][::-1]
                X_star_feed = X_star[np.newaxis, :, :]
                Qt = model.predict(X_star_feed)[0]
                X_hat[pre_step_num + t, :] = Qt.copy()
            else:
                # Fully missing: LSTM-only prediction (UNCHANGED path)
                X_star = X_hat[pre_step_num + t - time_lags, :][::-1]
                X_star_feed = X_star[np.newaxis, :, :]
                Qt = model.predict(X_star_feed)[0]
                X_hat[pre_step_num + t, :] = Qt.copy()
                alpha_log.append(1.0)  # forced temporal-only

        mat_pred[:, t] = np.matmul(W, X_hat[pre_step_num + t, :])

        if (t + 1) % 1000 == 0:
            print('Time step: %d, time cost: %d s, mean alpha: %.3f' % (
                t + 1, time.time() - start_time, np.mean(alpha_log[-1000:])))
            start_time = time.time()

    # Final step calibration (UNCHANGED logic, MODIFIED call)
    sparse_vec = sparse_mat[:, -1]
    init_final = {
        "W": W,
        "X": X_hat[dim2 + pre_step_num - np.max(time_lags) - 1:, :],
        "model": model
    }
    X_c, _ = OnlineLSTMReMF_Adaptive(
        sparse_vec, init_final, lambda_x / dim2, time_lags,
        gate_model=gate_model, binary_history=binary_history,
        ADJ=ADJ, t_idx=bin_buf_size - 1, y_prev=y_prev, y_pprev=y_pprev
    )
    X_hat[dim2 + pre_step_num - 1, :] = X_c.copy()
    mat_rec = np.matmul(W, X_hat[pre_step_num:, :].T)

    print(f"\n📊 Alpha statistics across test set:")
    print(f"   Mean alpha (temporal weight): {np.mean(alpha_log):.3f}")
    print(f"   Std alpha:                    {np.std(alpha_log):.3f}")
    print(f"   Min / Max:                    {np.min(alpha_log):.3f} / {np.max(alpha_log):.3f}")

    return np.round(mat_rec), np.round(mat_pred), np.array(alpha_log)

## Training

In [ ]:
rank = 60
maxiter = 200
eta = 0.2
lambda_w = 100
lambda_x = 100
lambda_g = 100
sampling_rate = 1.0
time_lags = np.array([1, 2, 288])
track = True
patience = 10

dim1, dim2 = training_set.shape
init = {
    "W": 0.1 * np.random.rand(dim1, rank),
    "X": 0.1 * np.random.rand(dim2, rank)
}

mat_hat, W, X, model, gate_model = LSTM_GL_ReMF_Adaptive(
    training_set, ADJ, init, time_lags,
    lambda_w, lambda_g, lambda_x, eta,
    sampling_rate, maxiter, track,
    patience, training_ground_truth,
    resume=True,
    gate_train_freq=10,
    gate_epochs=5,
    window=30
)

## Online Prediction with Adaptive Gate

In [ ]:
import time
start_time = time.time()

init_test = {
    "W": W,
    "X": X[-np.max(time_lags):, :],
    "model": model
}

test_mat_rec, test_mat_pred, alpha_log = online_prediction_adaptive(
    test_set, init_test, time_lags,
    lambda_w, 6000 * lambda_x, eta, maxiter,
    gate_model=gate_model,
    ADJ=ADJ,
    window=30
)

print('\nTotal test time: %.1f s' % (time.time() - start_time))
print('Shape of imputed data:', test_mat_rec.shape)
print('Shape of predicted data:', test_mat_pred.shape)

## Evaluation

In [ ]:
# Prediction error
pos = np.where(test_ground_truth != 0)
testPred_rmse = root_mean_squared_error(test_ground_truth, test_mat_pred, pos)
testPred_mape = mean_absolute_percentage_error(test_ground_truth, test_mat_pred, pos)
print('Test prediction RMSE: %.2f' % testPred_rmse)
print('Test prediction MAPE: %.2f%%' % testPred_mape)

print()

# Online imputation error
pos = np.where((test_set == 0) & (test_ground_truth != 0))
testRec_rmse = root_mean_squared_error(test_ground_truth, test_mat_rec, pos)
testRec_mape = mean_absolute_percentage_error(test_ground_truth, test_mat_rec, pos)
print('Test imputation RMSE: %.2f' % testRec_rmse)
print('Test imputation MAPE: %.2f%%' % testRec_mape)

## NEW: Alpha Analysis — Interpretability of Adaptive Weighting

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Alpha over time
axes[0].plot(alpha_log, color='steelblue', linewidth=0.8, alpha=0.8)
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Neutral (0.5)')
axes[0].set_xlabel('Test time step')
axes[0].set_ylabel('Alpha (temporal weight)')
axes[0].set_title('Adaptive Gate Output Over Time')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Alpha distribution
axes[1].hist(alpha_log, bins=40, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Alpha value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Adaptive Weights')

plt.tight_layout()
plt.savefig('alpha_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Alpha plot saved.')

## Prediction Visualization (unchanged)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig = plt.figure(figsize=(15, 4))
plt.style.use('classic')
ax = fig.add_axes([0.18, 0.20, 0.85, 0.75])
ax.set_facecolor('white')

sensor = 45
start_time = 1
end_time = 1441
array = test_set[sensor, start_time:end_time]

plt.plot(test_ground_truth[sensor, start_time:end_time],
         label='Ground Truth', color='blue', linewidth=1)
plt.plot(test_mat_pred[sensor, start_time:end_time],
         label='Predicted (Adaptive)', color='red', linewidth=1)

pos = np.where(array == 0)
zero = [[]]; seg = 0
for i in range(len(pos[0])):
    if (i == len(pos[0]) - 1) or (pos[0][i + 1] == pos[0][i] + 1):
        zero[seg].append(pos[0][i])
    if (i != len(pos[0]) - 1) and (pos[0][i + 1] != pos[0][i] + 1):
        zero[seg].append(pos[0][i]); zero.append([]); seg += 1

NM = [s for s in zero if len(s) > 1]
for i in range(len(NM)):
    ax.add_patch(patches.Rectangle((NM[i][0] - 1, 0), NM[i][-1] - NM[i][0] + 1,
                                     200, alpha=0.1, facecolor='yellow'))

plt.ylim(0, 75)
plt.xlim(0, 1441)
ax.set_ylabel("Speed Km/h")
plt.xticks(np.arange(0, 1440, 288),
           ["2015.Dec.27", "2015.Dec.28", "2015.Dec.29", "2015.Dec.30", "2015.Dec.31"])
plt.legend(loc='best')
plt.savefig(f'prediction_adaptive_seattle_{sensor}.png', bbox_inches='tight')
plt.show()

# License

<div class="alert alert-block alert-danger">
<b>This work is released under the MIT license.</b>
Base model: LSTM-GL-ReMF by Jinming Yang (SJTU). Adaptive weighting extension added per the methodology described in the attached research paper.
</div>